# Pavan et al. (2014) Index-Policy Benchmark -- `cont_single`

Estimates the **optimal expected revenue** for the bandit auction (2 buyers, 1 good, T=5, delta=0.9).

**Two estimates reported:**

- **Myerson (static)**: per-period Myerson VV. Not dynamic-IC; shown for reference only.
- **Pavan Gittins** (USE THIS): PST 2014 Eq. (15), initial hazard rate fixed at t=0,
  Gittins stopping-time index. Valid when I_{i,t,s}=1 in value space,
  which holds approximately for types near the centre of [0,5].

**Why eval_pavan_mechanism is NOT used:**
It evaluates (1-F_0(v_it))/f_0(v_it) using the initial distribution N(2.5,0.1)
at the CURRENT type v_it. After a buyer is allocated with sigma=1.0 at t=0,
their type can drift by ~1 unit in value space. At v=1.5, F_0(1.5)~=0 and f_0(1.5)~=0,
so hazard -> inf and J_i -> -inf, incorrectly withholding the good.
The correct PST formula requires the conditional hazard given history;
for the bandit auction this simplifies to inv_hz at the INITIAL type (fixed at t=0),
which is exactly what eval_pavan_bandit_auction does.

In [ ]:
RUN_ENV = 'colab'    # 'colab' | 'local'
LOCAL_PROJECT_DIR = None
# LOCAL_PROJECT_DIR = r'E:\Research\code\Dynamic Mechanism Design\Bandit Auction'

In [ ]:
import os, sys, subprocess

if RUN_ENV == 'colab':
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
        'jax', 'jaxlib', 'jax-cuda12-plugin', 'jax-cuda12-pjrt'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU',
        'jax[cuda12]', 'dm-haiku', 'optax', 'flax'])
    print('Install done. IMPORTANT: Runtime -> Restart session, then re-run from this cell.')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        PROJECT_DIR = '/content/drive/MyDrive/My Research/Dynamic Mechanism Design/Bandit Auction'
    except Exception:
        PROJECT_DIR = os.getcwd()
else:
    PROJECT_DIR = LOCAL_PROJECT_DIR or os.getcwd()

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## Imports & Auction Setup

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from bandit_auction.envs.cont_single import create_cont_single_auction
from bandit_auction.baselines.pavan import (
    eval_myerson_mechanism,
    eval_pavan_bandit_auction,
    eval_pavan_mechanism,
)

SEED = 42
print(f'JAX devices: {jax.devices()}')

In [ ]:
# sigma_no=0.0: non-allocated types exactly frozen — satisfies PST bandit auction assumption.
auction = create_cont_single_auction(
    n_buyers       = 2,
    n_goods        = 1,
    horizon        = 5,
    discount       = 0.9,
    type_mean      = 2.5,
    type_std       = 0.1,
    types_low      = 0.0,
    types_high     = 5.0,
    sigma_no       = 0.0,
    trans_lambda   = 1.0,
    allocation_cost= 0.1,
    seed           = SEED,
)
print(f'Auction: {auction.num_buyers}B x {auction.num_goods}G x T={auction.num_horizons}, '
      f'discount={auction.discount}')
print(f'sigma_no={auction.trans_sigma_no}')

## Myerson (Static) Baseline

Uses the per-period Myerson virtual value psi_i(v)=v-(1-F(v))/f(v) at the current type,
ignoring dynamic impulse-response terms.

**Note:** does *not* satisfy dynamic IC, so it is **not** a lower bound on IC-feasible
dynamic revenue. Reported for reference only.

In [ ]:
NUM_SAMPLES_MYERSON = 2000

print(f'Running Myerson baseline ({NUM_SAMPLES_MYERSON} trajectories)...')
myerson_rev, myerson_surplus = eval_myerson_mechanism(
    auction,
    num_samples = NUM_SAMPLES_MYERSON,
    seed        = SEED,
    mean        = 2.5,
    std         = 0.1,
)
print(f'  Revenue : {myerson_rev:.4f}')
print(f'  Surplus : {myerson_surplus:.4f}')

## Pavan Gittins Benchmark

Implements PST 2014 Eq. (15):
- 1/eta_{i0}(theta_{i0}) computed **once at t=0**, fixed thereafter.
- Gittins stopping-time index with flow payoff (theta_it - c_i - inv_hz_i).
- Revenue = E[sum_t delta^t (theta_it - inv_hz_i - c_i) x_it].

**Valid when** I_{i,t,s}=1 in value space (allocated) and I=0 (non-allocated).
With sigma_no=0 the I=0 condition holds exactly. The I=1 condition holds
approximately when types remain near 2.5; may overestimate if types drift far.

`num_index_samples` controls MC rollouts per Gittins index evaluation.

In [ ]:
NUM_SAMPLES_PAVAN   = 2000  # more trajectories for tighter MC error (~0.07 std)
NUM_INDEX_SAMPLES   = 200   # more rollouts per Gittins index evaluation

print(f'Running Pavan Gittins benchmark ({NUM_SAMPLES_PAVAN} trajectories, '
      f'{NUM_INDEX_SAMPLES} index samples)...')
pavan_rev, pavan_surplus = eval_pavan_bandit_auction(
    auction,
    num_samples       = NUM_SAMPLES_PAVAN,
    num_index_samples = NUM_INDEX_SAMPLES,
    seed              = SEED,
    mean              = 2.5,
    std               = 0.1,
)
print(f'  Revenue : {pavan_rev:.4f}')
print(f'  Surplus : {pavan_surplus:.4f}')

## Summary

In [ ]:
print('='*55)
print(f'  Benchmark revenue estimates (cont_single, seed={SEED})')
print('='*55)
print(f'  Myerson (static, not dynamic-IC):  {myerson_rev:.4f}')
print(f'  Pavan Gittins  (USE THIS)       :  {pavan_rev:.4f}')
print()
print(f'  PROFIT_GOAL for cont_single_colab.ipynb -> {pavan_rev:.4f}')
print(f'  (MC std error ~ {pavan_rev*0.007:.3f} with {NUM_SAMPLES_PAVAN} samples)')

## (Optional) Compare with Trained Mechanism

Load a trained mechanism from `cont_single_colab.ipynb` and compare its revenue
against the Pavan benchmark.

In [ ]:
import pickle
from bandit_auction.core import AuctionSimulator, BoundedContPolicy, build_bounded_policy_net
from bandit_auction.mechanisms.network import Mechanism, build_mechanism_network
from bandit_auction.train.evaluate import eval_mechanism_multikey, print_eval_report

EXPERIMENT = 'cont_single'
METHOD     = 'acceptance'   # 'acceptance' | 'penalty'
LOAD_SEED  = 42

load_dir = f'outputs/{EXPERIMENT}/{METHOD}/seed{LOAD_SEED}'

if not os.path.exists(f'{load_dir}/mechanism_params.pkl'):
    print(f'No saved params at {load_dir}/ — run cont_single_colab.ipynb first.')
else:
    with open(f'{load_dir}/mechanism_params.pkl', 'rb') as f:
        mechanism_params = pickle.load(f)
    with open(f'{load_dir}/policy_params.pkl', 'rb') as f:
        policy_params = pickle.load(f)

    mechanism_network = build_mechanism_network()
    policy_network = build_bounded_policy_net(
        d_priv=auction.d_priv, d_pub=auction.d_pub,
        G=auction.num_goods, hidden=128, residual_init_std=1.0,
    )

    eval_keys = jax.random.split(jax.random.PRNGKey(88888), 32)
    results = eval_mechanism_multikey(
        auction           = auction,
        mechanism_network = mechanism_network,
        mechanism_params  = mechanism_params,
        profit_goal       = pavan_rev,   # use Pavan revenue as the goal
        num_samples       = 256,
        keys              = eval_keys,
        policy_network    = policy_network,
        policy_params     = policy_params,
        is_discrete       = False,
    )
    print_eval_report(results, prefix=f'{EXPERIMENT} ({METHOD}): ')
    print(f'  Pavan benchmark revenue : {pavan_rev:.4f}')
    print(f'  Trained mechanism revenue: {results["profit_stats"]["mean"]:.4f}')
    gap = pavan_rev - results['profit_stats']['mean']
    print(f'  Gap to benchmark        : {gap:+.4f}')

---

# Pavan et al. (2014) Gittins Benchmark — `discrete_single`

Estimates the **optimal expected revenue** for the bandit auction with **discrete types** (2 buyers, 1 good, T=5, δ=0.9, types ∈ {1,2,3}).

**Two estimates reported:**

- **Static Myerson** (`eval_pavan_discrete`): per-period static Myerson virtual value applied at the *current* type. Not dynamically IC; shown for reference only.
- **Pavan Gittins** (USE THIS) (`eval_pavan_bandit_auction_discrete`): PST 2014 Section 5 discrete analogue. Fixes the inverse hazard rate at the **initial type** and uses the Gittins stopping-time index. This is the correct dynamic benchmark.

### Math: Discrete Inverse Hazard Rate

For a uniform prior over K sorted types $\{v^0,\ldots,v^{K-1}\}$, the Myerson discrete virtual valuation is:
$$\psi(v^k) = v^k - \frac{1-F(v^k)}{\pi_k}(v^{k+1}-v^k)$$
where $F(v^k) = (k+1)/K$ and $\pi_k = 1/K$, giving $(1-F(v^k))/\pi_k = K-1-k$. The information-rent coefficient (discrete inv_hz) is:
$$\mathrm{inv\_hz}(k) = (K-1-k)(v^{k+1}-v^k)$$
This is fixed at the **initial type** $k_0$ for the entire episode (same as $(1-F_0)/f_0$ in the continuous case).

For types $\{1,2,3\}$ (K=3, step=1): $\mathrm{inv\_hz} = \{2, 1, 0\}$ for initial types $\{1, 2, 3\}$.

The virtual flow payoff is $\phi_i(v_{it}, k_{i,0}) = v_{it} - c_i - \mathrm{inv\_hz}(k_{i,0})$, and revenue uses the PST identity:
$$\text{Revenue} = \mathbb{E}\!\left[\sum_t \delta^t \sum_i \phi_i(v_{it},k_{i,0})\, x_{it}\right]$$

**Validity:** Same impulse-response approximation ($I_{0,t}\approx 1$ for allocated buyers) as the continuous case. The discrete exponential-decay kernel concentrates near the current type when allocated (decay_rate=0.5), making the initial type informative about the future.

## Discrete Auction Setup

In [ ]:
from bandit_auction.envs.discrete_single import create_discrete_single_auction
from bandit_auction.baselines.pavan import (
    eval_pavan_discrete,
    eval_pavan_bandit_auction_discrete,
)

SEED_DISC = 42
TYPE_VALUES = [1.0, 2.0, 3.0]

auction_disc = create_discrete_single_auction(
    n_buyers       = 2,
    n_goods        = 1,
    horizon        = 5,
    discount       = 0.9,
    type_values    = TYPE_VALUES,
    allocation_cost= 0.1,
    seed           = SEED_DISC,
)

type_vals = auction_disc.buyer_types[0, :, 0]
K = len(type_vals)
print(f'Auction: {auction_disc.num_buyers}B x {auction_disc.num_goods}G x T={auction_disc.num_horizons}, '
      f'discount={auction_disc.discount}')
print(f'Type values: {list(float(v) for v in type_vals)}')
print()

# Print the discrete inv_hz table for reference
print('Discrete inv_hz(k) for uniform prior:')
for k in range(K):
    if k < K - 1:
        step = float(type_vals[k+1]) - float(type_vals[k])
        inv_hz = (K - 1 - k) * step
    else:
        inv_hz = 0.0
    vv = float(type_vals[k]) - inv_hz
    print(f'  k={k}, v={float(type_vals[k]):.1f}: inv_hz={inv_hz:.2f}, '
          f'psi(v)={vv:.2f}, phi(v)=psi(v)-c={vv-0.1:.2f}')

## Static Myerson Baseline (discrete, per-period)

Uses the current-type virtual value $\psi(v^k)$ at each period independently. **Not dynamically IC**; shown for reference only. This is what `eval_pavan_discrete` computes.

In [ ]:
NUM_SAMPLES_DISC_MYERSON = 2000

print(f'Running static Myerson baseline ({NUM_SAMPLES_DISC_MYERSON} trajectories)...')
disc_myerson_rev, disc_myerson_surplus = eval_pavan_discrete(
    auction_disc,
    num_samples = NUM_SAMPLES_DISC_MYERSON,
    seed        = SEED_DISC,
)
print(f'  Revenue : {disc_myerson_rev:.4f}')
print(f'  Surplus : {disc_myerson_surplus:.4f}')

## Pavan Gittins Benchmark (discrete, USE THIS)

Discrete analogue of PST 2014 Section 5 Eq. (15):
- `inv_hz_disc(k_0)` computed **once at t=0** from the initial type, fixed thereafter.
- Gittins stopping-time index with flow payoff $\phi_i(v_{it}) = v_{it} - c_i - \mathrm{inv\_hz}_i$.
- Revenue = $\mathbb{E}[\sum_t \delta^t \sum_i \phi_i(v_{it}) x_{it}]$.

`num_index_samples` controls MC rollouts per Gittins index evaluation. Higher values reduce variance in the index estimate.

In [ ]:
NUM_SAMPLES_DISC_PAVAN  = 2000
NUM_INDEX_SAMPLES_DISC  = 200

print(f'Running Pavan Gittins benchmark ({NUM_SAMPLES_DISC_PAVAN} trajectories, '
      f'{NUM_INDEX_SAMPLES_DISC} index samples)...')
disc_pavan_rev, disc_pavan_surplus = eval_pavan_bandit_auction_discrete(
    auction_disc,
    num_samples       = NUM_SAMPLES_DISC_PAVAN,
    num_index_samples = NUM_INDEX_SAMPLES_DISC,
    seed              = SEED_DISC,
)
print(f'  Revenue : {disc_pavan_rev:.4f}')
print(f'  Surplus : {disc_pavan_surplus:.4f}')

## Summary (discrete_single)

In [ ]:
print('='*60)
print(f'  Benchmark revenue estimates (discrete_single, seed={SEED_DISC})')
print('='*60)
print(f'  Static Myerson (not dynamic-IC):  {disc_myerson_rev:.4f}')
print(f'  Pavan Gittins  (USE THIS)       :  {disc_pavan_rev:.4f}')
print()
print(f'  PROFIT_GOAL for discrete_single_colab.ipynb -> {disc_pavan_rev:.4f}')
print()
print('  inv_hz values at initial type (for reference):')
for k in range(K):
    if k < K - 1:
        step = float(type_vals[k+1]) - float(type_vals[k])
        inv_hz_k = (K - 1 - k) * step
    else:
        inv_hz_k = 0.0
    print(f'    v_0={float(type_vals[k]):.1f} -> inv_hz={inv_hz_k:.2f}')

## (Optional) Compare with Trained Mechanism (discrete_single)

Load a trained mechanism from `discrete_single_colab.ipynb` and compare against the Pavan Gittins benchmark.

In [ ]:
import pickle
from bandit_auction.core import AuctionSimulator, build_discrete_policy_net
from bandit_auction.mechanisms.network import Mechanism, build_mechanism_network
from bandit_auction.train.evaluate import eval_mechanism_multikey, print_eval_report

EXPERIMENT_DISC = 'discrete_single'
METHOD_DISC     = 'acceptance'   # 'acceptance' | 'penalty'
LOAD_SEED_DISC  = 42

load_dir_disc = f'outputs/{EXPERIMENT_DISC}/{METHOD_DISC}/seed{LOAD_SEED_DISC}'

if not os.path.exists(f'{load_dir_disc}/mechanism_params.pkl'):
    print(f'No saved params at {load_dir_disc}/ — run discrete_single_colab.ipynb first.')
else:
    with open(f'{load_dir_disc}/mechanism_params.pkl', 'rb') as f:
        mechanism_params_disc = pickle.load(f)
    with open(f'{load_dir_disc}/policy_params.pkl', 'rb') as f:
        policy_params_disc = pickle.load(f)

    mechanism_network_disc = build_mechanism_network()
    policy_network_disc = build_discrete_policy_net(
        d_priv=auction_disc.d_priv,
        d_pub=auction_disc.d_pub,
    )

    eval_keys_disc = jax.random.split(jax.random.PRNGKey(88888), 32)
    results_disc = eval_mechanism_multikey(
        auction           = auction_disc,
        mechanism_network = mechanism_network_disc,
        mechanism_params  = mechanism_params_disc,
        profit_goal       = disc_pavan_rev,
        num_samples       = 256,
        keys              = eval_keys_disc,
        policy_network    = policy_network_disc,
        policy_params     = policy_params_disc,
        is_discrete       = True,
    )
    print_eval_report(results_disc, prefix=f'{EXPERIMENT_DISC} ({METHOD_DISC}): ')
    print(f'  Pavan Gittins benchmark : {disc_pavan_rev:.4f}')
    print(f'  Trained mechanism profit: {results_disc["profit_stats"]["mean"]:.4f}')
    gap = disc_pavan_rev - results_disc['profit_stats']['mean']
    print(f'  Gap to benchmark        : {gap:+.4f}')